In [9]:
import pandas as pd
from pathlib import Path

charging_path = Path("home_charging_clean.csv")
grid_path = Path("combined_hourly_clean.csv")

# Load the full home charging dataset
sessions = pd.read_csv(
    charging_path,
    usecols=["session_id", "start_time", "end_time", "kwh", "home_or_public"],
    parse_dates=["start_time", "end_time"]
)

# Load hourly grid and weather data
grid = pd.read_csv(
    grid_path,
    usecols=["local_time", "grid_utc", "demand_mwh", "temp_f"],
    parse_dates=["local_time", "grid_utc"]
)

# Load the CSV containing flagged sessions
flagged = pd.read_csv("battery_capacity_issues.csv")

# Identify matching session IDs
flagged_ids = set(flagged["session_id"])
matches = sessions["session_id"].isin(flagged_ids)

print("Flagged IDs:", len(flagged_ids))
print("Matching sessions:", matches.sum())
print("Unmatched flagged IDs:", len(flagged_ids - set(sessions.loc[matches, "session_id"])))
print("Duplicate session IDs:", sessions["session_id"].duplicated().sum())

# Exclude flagged sessions
sessions_clean = sessions.loc[~matches].copy()

print("Sessions remaining:", len(sessions_clean))



Flagged IDs: 20
Matching sessions: 20
Unmatched flagged IDs: 0
Duplicate session IDs: 0
Sessions remaining: 2447974


In [10]:
# Identify the four daylight-saving transition dates
dst_dates = pd.to_datetime([
    "2022-03-13",
    "2022-11-06",
    "2023-03-12",
    "2023-11-05"
])

# Identify sessions with invalid durations
invalid_duration = (
    sessions_clean["end_time"] <= sessions_clean["start_time"]
)

# Identify sessions occurring on DST transition dates
on_dst_date = (
    sessions_clean["start_time"].dt.normalize().isin(dst_dates)
    | sessions_clean["end_time"].dt.normalize().isin(dst_dates)
)

# Exclude invalid sessions outside DST dates
remove_invalid = invalid_duration & ~on_dst_date

sessions_valid = sessions_clean.loc[~remove_invalid].copy()

# Verify the result
print("Invalid sessions excluded:", remove_invalid.sum())
print("Sessions remaining:", len(sessions_valid))

Invalid sessions excluded: 21
Sessions remaining: 2447953


In [11]:
from datetime import timedelta

# Example charging session
start = pd.Timestamp("2022-09-01 20:30:00")
end = pd.Timestamp("2022-09-01 22:00:00")
energy = 9.0

# Calculate duration and average charging power
duration = (end - start).total_seconds() / 3600
avg_power = energy / duration

print("Duration (hours):", duration)
print("Average power (kW):", avg_power)

# Find the beginning of the first hour
current_hour = start.floor("h")

# Store the hourly allocations
hourly_energy = []

while current_hour < end:
    next_hour = current_hour + timedelta(hours=1)

    # Determine the portion of the session within this hour
    overlap_start = max(start, current_hour)
    overlap_end = min(end, next_hour)

    overlap_hours = (
        overlap_end - overlap_start
    ).total_seconds() / 3600

    # Allocate energy based on time spent charging
    allocated_kwh = avg_power * overlap_hours

    hourly_energy.append({
        "hour": current_hour,
        "kwh": allocated_kwh
    })

    current_hour = next_hour

display(pd.DataFrame(hourly_energy))

print("Original energy:", energy)
print("Allocated energy:", sum(row["kwh"] for row in hourly_energy))

Duration (hours): 1.5
Average power (kW): 6.0


,hour,kwh
0,2022-09-01 20:00:00,3.0
1,2022-09-01 21:00:00,6.0


Original energy: 9.0
Allocated energy: 9.0


In [12]:
def allocate_hourly(start, end, energy):
    # Calculate session duration in hours
    duration = (end - start).total_seconds() / 3600

    if duration <= 0:
        raise ValueError("Session duration must be positive")

    # Calculate average charging power
    avg_power = energy / duration

    # Begin at the first overlapping hour
    current_hour = start.floor("h")
    hourly_energy = []

    while current_hour < end:
        next_hour = current_hour + pd.Timedelta(hours=1)

        overlap_start = max(start, current_hour)
        overlap_end = min(end, next_hour)

        overlap_hours = (
            overlap_end - overlap_start
        ).total_seconds() / 3600

        allocated_kwh = avg_power * overlap_hours

        hourly_energy.append({
            "hour": current_hour,
            "kwh": allocated_kwh
        })

        current_hour = next_hour

    return hourly_energy

test_result = allocate_hourly(
    pd.Timestamp("2022-09-01 20:30:00"),
    pd.Timestamp("2022-09-01 22:00:00"),
    9.0
)

display(pd.DataFrame(test_result))

print(
    "Total allocated:",
    sum(row["kwh"] for row in test_result)
)

,hour,kwh
0,2022-09-01 20:00:00,3.0
1,2022-09-01 21:00:00,6.0


Total allocated: 9.0


In [13]:
# Define daylight-saving transition windows
dst_windows = [
    ("2022-03-13", "02:00", "03:00"),
    ("2022-11-06", "01:00", "02:00"),
    ("2023-03-12", "02:00", "03:00"),
    ("2023-11-05", "01:00", "02:00")
]

# Initially, no sessions are marked as ambiguous
dst_ambiguous = pd.Series(False, index=sessions_valid.index)

for date, start_hour, end_hour in dst_windows:
    window_start = pd.Timestamp(f"{date} {start_hour}")
    window_end = pd.Timestamp(f"{date} {end_hour}")

    # Identify sessions overlapping the transition window
    affected = (
        (sessions_valid["start_time"] < window_end) &
        (sessions_valid["end_time"] > window_start)
    )

    # Include invalid-duration sessions on transition dates
    invalid_on_date = (
        (sessions_valid["end_time"] <= sessions_valid["start_time"]) &
        (
            (sessions_valid["start_time"].dt.normalize() == pd.Timestamp(date)) |
            (sessions_valid["end_time"].dt.normalize() == pd.Timestamp(date))
        )
    )

    dst_ambiguous = dst_ambiguous | affected | invalid_on_date

print("DST-ambiguous sessions:", dst_ambiguous.sum())
# Exclude sessions flagged as DST-ambiguous
sessions_ready = sessions_valid.loc[~dst_ambiguous].copy()

print("DST-ambiguous sessions excluded:", dst_ambiguous.sum())
print("Sessions remaining:", len(sessions_ready))

# Verify that no flagged sessions remain
print(
    "Flagged sessions remaining:",
    sessions_ready.index.isin(
        sessions_valid.index[dst_ambiguous]
    ).sum()
)

DST-ambiguous sessions: 2873
DST-ambiguous sessions excluded: 2873
Sessions remaining: 2445080
Flagged sessions remaining: 0


In [14]:
# Exclude DST-ambiguous sessions
sessions_ready = sessions_valid.loc[~dst_ambiguous].copy()

print("Sessions remaining:", len(sessions_ready))

Sessions remaining: 2445080


In [15]:
hourly_totals = {}

for session in sessions_ready.itertuples(index=False):
    allocated_hours = allocate_hourly(session.start_time, session.end_time, session.kwh)

    for allocation in allocated_hours:
        if (allocation["hour"].normalize() in dst_dates):
            continue
        elif (allocation["hour"] in hourly_totals):
            hourly_totals[allocation["hour"]] += allocation["kwh"]
        else:
            hourly_totals[allocation["hour"]] = allocation["kwh"]
print("Unique hours:", len(hourly_totals))
print("Total allocated energy:", sum(hourly_totals.values()))

Unique hours: 11566
Total allocated energy: 32566999.861260578


In [16]:
hourly_homeCharging_demand = pd.DataFrame(list(hourly_totals.items()), columns=["local_time", "home_charging_kwh"])
hourly_homeCharging_demand = hourly_homeCharging_demand.sort_values(by="local_time")

hourly_homeCharging_demand = hourly_homeCharging_demand.reset_index(drop=True)
display(hourly_homeCharging_demand.head())

,local_time,home_charging_kwh
0,2022-09-01 14:00:00,5.287614
1,2022-09-01 15:00:00,3.563183
2,2022-09-01 16:00:00,4.554365
3,2022-09-01 17:00:00,7.773969
4,2022-09-01 18:00:00,25.747746


In [17]:

hourly_chargeDemandTemp = grid.copy()
start_date = pd.Timestamp("2022-09-01 14:00:00")
end_date = pd.Timestamp("2023-12-31 23:00:00")

filtered_hourly_chargeDemandTemp = hourly_chargeDemandTemp[(hourly_chargeDemandTemp["local_time"] >= start_date) & (hourly_chargeDemandTemp["local_time"] <= end_date)] 
filtered_hourly_chargeDemandTemp = filtered_hourly_chargeDemandTemp[~hourly_chargeDemandTemp["local_time"].dt.normalize().isin(dst_dates)]


print(type(filtered_hourly_chargeDemandTemp))
print(filtered_hourly_chargeDemandTemp.shape)
print(filtered_hourly_chargeDemandTemp.head())

merged_hourly = pd.merge(filtered_hourly_chargeDemandTemp, hourly_homeCharging_demand, on="local_time", how="left")


print(merged_hourly["home_charging_kwh"].isna().sum())

merged_hourly["home_charging_kwh"] = merged_hourly["home_charging_kwh"].fillna(0)

print(merged_hourly["home_charging_kwh"].isna().sum())


display(merged_hourly.head())

<class 'pandas.DataFrame'>
(11602, 4)
      demand_mwh                  grid_utc  temp_f          local_time
5845         981 2022-09-01 20:00:00+00:00    88.9 2022-09-01 14:00:00
5846        1000 2022-09-01 21:00:00+00:00    88.2 2022-09-01 15:00:00
5847         997 2022-09-01 22:00:00+00:00    87.1 2022-09-01 16:00:00
5848        1009 2022-09-01 23:00:00+00:00    86.9 2022-09-01 17:00:00
5849        1002 2022-09-02 00:00:00+00:00    85.6 2022-09-01 18:00:00
51
0


C:\Users\pengo\AppData\Local\Temp\ipykernel_26432\730523494.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_hourly_chargeDemandTemp = filtered_hourly_chargeDemandTemp[~hourly_chargeDemandTemp["local_time"].dt.normalize().isin(dst_dates)]


,demand_mwh,grid_utc,temp_f,local_time,home_charging_kwh
0,981,2022-09-01 20:00:00+00:00,88.9,2022-09-01 14:00:00,5.287614
1,1000,2022-09-01 21:00:00+00:00,88.2,2022-09-01 15:00:00,3.563183
2,997,2022-09-01 22:00:00+00:00,87.1,2022-09-01 16:00:00,4.554365
3,1009,2022-09-01 23:00:00+00:00,86.9,2022-09-01 17:00:00,7.773969
4,1002,2022-09-02 00:00:00+00:00,85.6,2022-09-01 18:00:00,25.747746


In [18]:
overnight_observations = merged_hourly.copy()


hours = overnight_observations["local_time"].dt.hour

filtered_overnight_observations = overnight_observations[(hours >= 20) | (hours < 6)]

print("Overnight observations:", len(filtered_overnight_observations))

print(
    "Hours included:",
    sorted(filtered_overnight_observations["local_time"].dt.hour.unique())
)

display(filtered_overnight_observations.head())


Overnight observations: 4834
Hours included: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


,demand_mwh,grid_utc,temp_f,local_time,home_charging_kwh
6,963,2022-09-02 02:00:00+00:00,83.8,2022-09-01 20:00:00,36.801465
7,905,2022-09-02 03:00:00+00:00,81.1,2022-09-01 21:00:00,33.986773
8,830,2022-09-02 04:00:00+00:00,79.8,2022-09-01 22:00:00,63.697259
9,747,2022-09-02 05:00:00+00:00,76.6,2022-09-01 23:00:00,72.290366
10,608,2022-09-02 06:00:00+00:00,75.1,2022-09-02 00:00:00,102.105306


In [19]:
quartile_observations = filtered_overnight_observations.copy()

home_kwh = quartile_observations["home_charging_kwh"]

charging_quartiles = pd.qcut(home_kwh, q=4, labels=["Q1", "Q2", "Q3", "Q4"])

quartile_observations["charging_quartile"] = charging_quartiles

print(quartile_observations["charging_quartile"].value_counts().sort_index())

display(
    quartile_observations[
        ["local_time", "home_charging_kwh", "charging_quartile"]
    ].head()
)

charging_quartile
Q1    1209
Q2    1208
Q3    1208
Q4    1209
Name: count, dtype: int64


,local_time,home_charging_kwh,charging_quartile
6,2022-09-01 20:00:00,36.801465,Q1
7,2022-09-01 21:00:00,33.986773,Q1
8,2022-09-01 22:00:00,63.697259,Q1
9,2022-09-01 23:00:00,72.290366,Q1
10,2022-09-02 00:00:00,102.105306,Q1


In [20]:
q1 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q1"]
q2 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q2"]
q3 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q3"]
q4 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q4"]

display(q1["demand_mwh"].mean())
display(q2["demand_mwh"].mean())
display(q3["demand_mwh"].mean())
display(q4["demand_mwh"].mean())

difference = q4["demand_mwh"].mean() - q1["demand_mwh"].mean()

print(f"Q4 - Q1 difference: {difference:.2f} MW")


np.float64(622.3655913978495)

np.float64(657.25)

np.float64(702.5372516556291)

np.float64(697.671629445823)

Q4 - Q1 difference: 75.31 MW


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# 1. Prepare regression data
regression_data = filtered_overnight_observations.copy()
regression_data = regression_data.sort_values("local_time")

# Add month and hour as categorical variables
regression_data["month"] = regression_data["local_time"].dt.month
regression_data["hour"] = regression_data["local_time"].dt.hour

# 2. Define input variables
X = regression_data[
    ["home_charging_kwh", "temp_f", "month", "hour"]
].copy()

# Encode month and hour as categorical features
X = pd.get_dummies(
    X,
    columns=["month", "hour"],
    drop_first=True,
    dtype=int
)

# Define the target variable
y = regression_data["demand_mwh"]

# 3. Split chronologically: 80% train, 20% validation
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_val = X.iloc[split:]

y_train = y.iloc[:split]
y_val = y.iloc[split:]

# 4. Train linear regression
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Generate predictions
train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

# 6. Evaluate performance
print("Training R²:", r2_score(y_train, train_predictions))
print("Validation R²:", r2_score(y_val, val_predictions))

print("Training MAE:", mean_absolute_error(y_train, train_predictions))
print("Validation MAE:", mean_absolute_error(y_val, val_predictions))

# 7. Examine charging coefficient
coefficients = pd.Series(model.coef_, index=X.columns)

print(
    "\nHome charging coefficient:",
    coefficients["home_charging_kwh"]
)

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))

Training R²: 0.7809784579289706
Validation R²: 0.7486288550419883
Training MAE: 53.884460765067004
Validation MAE: 44.155850176306636

Home charging coefficient: 0.006092758963755289
Training observations: 3867
Validation observations: 967


In [22]:
# Baseline model without home charging
X_baseline = X.drop(columns=["home_charging_kwh"])

# Use the exact same chronological split
X_base_train = X_baseline.iloc[:split]
X_base_val = X_baseline.iloc[split:]

baseline_model = LinearRegression()
baseline_model.fit(X_base_train, y_train)

baseline_predictions = baseline_model.predict(X_base_val)

# Compare validation performance
baseline_r2 = r2_score(y_val, baseline_predictions)
baseline_mae = mean_absolute_error(y_val, baseline_predictions)

charging_r2 = r2_score(y_val, val_predictions)
charging_mae = mean_absolute_error(y_val, val_predictions)

print("Without charging:")
print(f"Validation R²: {baseline_r2:.4f}")
print(f"Validation MAE: {baseline_mae:.2f} MW")

print("\nWith charging:")
print(f"Validation R²: {charging_r2:.4f}")
print(f"Validation MAE: {charging_mae:.2f} MW")

print(f"\nR² improvement: {charging_r2 - baseline_r2:.4f}")
print(f"MAE reduction: {baseline_mae - charging_mae:.2f} MW")

Without charging:
Validation R²: 0.7645
Validation MAE: 43.00 MW

With charging:
Validation R²: 0.7486
Validation MAE: 44.16 MW

R² improvement: -0.0158
MAE reduction: -1.15 MW


In [23]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

# Use the original training portion for cross-validation
tscv = TimeSeriesSplit(n_splits=4)

results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):

    # Same chronological fold for both models
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Baseline: no home charging
    base = LinearRegression()
    base.fit(
        X_fold_train.drop(columns=["home_charging_kwh"]),
        y_fold_train
    )

    base_pred = base.predict(
        X_fold_val.drop(columns=["home_charging_kwh"])
    )

    # Full model: includes home charging
    full = LinearRegression()
    full.fit(X_fold_train, y_fold_train)

    full_pred = full.predict(X_fold_val)

    results.append({
        "fold": fold,
        "baseline_r2": r2_score(y_fold_val, base_pred),
        "full_r2": r2_score(y_fold_val, full_pred),
        "baseline_mae": mean_absolute_error(y_fold_val, base_pred),
        "full_mae": mean_absolute_error(y_fold_val, full_pred),
        "charging_coefficient": full.coef_[
            X_train.columns.get_loc("home_charging_kwh")
        ],
        "train_r2": r2_score(
            y_fold_train,
            full.predict(X_fold_train)
        )
    })

cv_results = pd.DataFrame(results)

cv_results["r2_improvement"] = (
    cv_results["full_r2"] - cv_results["baseline_r2"]
)

cv_results["mae_reduction"] = (
    cv_results["baseline_mae"] - cv_results["full_mae"]
)

cv_results["train_val_gap"] = (
    cv_results["train_r2"] - cv_results["full_r2"]
)

display(cv_results.round(4))

print("\nMean results:")
print(cv_results.mean(numeric_only=True).round(4))

print("\nStandard deviations:")
print(cv_results.std(numeric_only=True).round(4))

,fold,baseline_r2,full_r2,baseline_mae,full_mae,charging_coefficient,train_r2,r2_improvement,mae_reduction,train_val_gap
0,1,-1.4819,-2.2448,182.2225,213.8556,-0.0129,0.7761,-0.7630,-31.6331,3.0210
1,2,0.4101,0.4166,81.1189,77.8273,-0.0087,0.8827,0.0065,3.2916,0.4661
2,3,-0.6366,-0.7935,150.5529,157.5605,-0.0055,0.8817,-0.1570,-7.0077,1.6753
3,4,0.4498,0.4688,97.7397,95.9888,0.0013,0.8023,0.0190,1.7509,0.3335



Mean results:
fold                      2.5000
baseline_r2              -0.3146
full_r2                  -0.5382
baseline_mae            127.9085
full_mae                136.3081
charging_coefficient     -0.0064
train_r2                  0.8357
r2_improvement           -0.2236
mae_reduction            -8.3996
train_val_gap             1.3740
dtype: float64

Standard deviations:
fold                     1.2910
baseline_r2              0.9266
full_r2                  1.2785
baseline_mae            46.7696
full_mae                61.9434
charging_coefficient     0.0060
train_r2                 0.0547
r2_improvement           0.3684
mae_reduction           16.1395
train_val_gap            1.2530
dtype: float64
